# Pendahuluan
Bagian ini menyiapkan dataset kedalam bentuk Graph, dan diekspor ke file agar bisa digunakan pada notebook lain dengan tasknya masing-masing
1. Preprocessing
   * Remove Duplicate Rows
   * Remove Nan
2. Build Basic Graph
3. Build Node Attribute
4. Drop Isolated Non-Fraud Nodes
5. Export to File.

Proses penyiapan dilakukan untuk kedua dataset dengan hasil export:
1. first_graph.pkl
2. second_graph.pkl

In [1]:
# Dataset Path
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/ethereum-transactions-for-fraud-detection/first_order_df.csv
/kaggle/input/ethereum-transactions-for-fraud-detection/second_order_df.csv


In [2]:
df1 = pd.read_csv('/kaggle/input/ethereum-transactions-for-fraud-detection/first_order_df.csv')
df1 = df1.sort_values('TimeStamp').reset_index(drop=True)
print('########## Dataset 1 ############')
print(df1.describe())

df2 = pd.read_csv('/kaggle/input/ethereum-transactions-for-fraud-detection/second_order_df.csv')
df2 = df2.sort_values('TimeStamp').reset_index(drop=True)
# print('\n########## Dataset 2 ############')
# print(df2.describe())

########## Dataset 1 ############
         Unnamed: 0   BlockHeight     TimeStamp          Value        isError
count  254973.00000  2.549730e+05  2.549730e+05  254973.000000  254973.000000
mean   127486.00000  5.391963e+06  1.522941e+09       4.784175       0.061316
std     73604.50943  7.332419e+05  1.107530e+07     207.403848       0.239910
min         0.00000  2.722620e+06  1.480521e+09       0.000000       0.000000
25%     63743.00000  4.908133e+06  1.515950e+09       0.000000       0.000000
50%    127486.00000  5.356399e+06  1.522521e+09       0.012500       0.000000
75%    191229.00000  5.855385e+06  1.529985e+09       0.500000       0.000000
max    254972.00000  7.781130e+06  1.558142e+09   25533.614518       1.000000


# 1. Preprocessing

## a. Cek Duplikasi Transaksi

In [3]:
# Cek Duplikasi pada row (Transaksi ethereum memiliki TxHash Unique)

def check_duplicate_transaction(df):
    # Cek duplikat pada kolom TxHash
    print("=== CEK DUPLIKAT TxHash ===")
    print(f"Total records: {len(df)}")
    print(f"Unique TxHash: {df['TxHash'].nunique()}")
    print(f"Duplicate count: {len(df) - df['TxHash'].nunique()}")
    print(f"Duplicate percentage: {(len(df) - df['TxHash'].nunique()) / len(df) * 100:.2f}%")
    
    # Tampilkan record yang duplicate
    duplicate_TxHashes = df[df.duplicated('TxHash', keep=False)]  # keep=False menunjukkan semua duplicate
    print(f"\nRecords dengan TxHash duplicate:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', None)
    
    # Tampilkan tabel dengan horizontal scroll
    display(duplicate_TxHashes[['TxHash', 'From', 'To', 'Value', 'TimeStamp', 'isError']].head(10))
    print()

check_duplicate_transaction(df1)
check_duplicate_transaction(df2)

=== CEK DUPLIKAT TxHash ===
Total records: 254973
Unique TxHash: 102149
Duplicate count: 152824
Duplicate percentage: 59.94%

Records dengan TxHash duplicate:


,TxHash,From,To,Value,TimeStamp,isError
0,0xfcae50a8fd6437d40ba50661d8f4566366b987a7d8359f4431985c96f2c3d678,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.511739,1480520845,0
1,0xfcae50a8fd6437d40ba50661d8f4566366b987a7d8359f4431985c96f2c3d678,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.511739,1480520845,0
2,0xfcae50a8fd6437d40ba50661d8f4566366b987a7d8359f4431985c96f2c3d678,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.511739,1480520845,0
3,0x5cd9d57ae3157f88a009c85f353d33647b760c78797276e037b67a880b3a3348,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.835023,1480521516,0
4,0x5cd9d57ae3157f88a009c85f353d33647b760c78797276e037b67a880b3a3348,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.835023,1480521516,0
5,0x5cd9d57ae3157f88a009c85f353d33647b760c78797276e037b67a880b3a3348,0x70faa28a6b8d6829a4b1e649d26ec9a2a39ba413,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,173.835023,1480521516,0
6,0x160c2343d67c384b7a3ad20200b6ba236265b8216f268df404c58741ac624a7c,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,0x03e9a232700c14fbc15b1187a734a7de2824ea97,347.346320,1480532847,0
7,0x160c2343d67c384b7a3ad20200b6ba236265b8216f268df404c58741ac624a7c,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,0x03e9a232700c14fbc15b1187a734a7de2824ea97,347.346320,1480532847,0
8,0x160c2343d67c384b7a3ad20200b6ba236265b8216f268df404c58741ac624a7c,0x9af3bf0b0a117d3fbfb37dfc7fa67f9a645488fc,0x03e9a232700c14fbc15b1187a734a7de2824ea97,347.346320,1480532847,0
9,0x6adb381699a43a495b278622fab84634014d9ffa79dc3d0c6244ec2cdc1701de,0x32be343b94f860124dc4fee278fdcbd38c102d88,0xbe2c434c49959ec0b6bb4885e63ff9977c009298,5.165700,1488662232,0



=== CEK DUPLIKAT TxHash ===
Total records: 6794521
Unique TxHash: 4604555
Duplicate count: 2189966
Duplicate percentage: 32.23%

Records dengan TxHash duplicate:


,TxHash,From,To,Value,TimeStamp,isError
5,0xaef899f6769fe611b5557457a3802d5cfae060027745f6ba9ac21d4648f02ca8,0xfc97c2c87e3cc03beb5fa2facaa10bbd0d0da1af,0x08cac8952641d8fc526ec1ab4f2df826a5e7710f,0.01000,1438920403,0
6,0xaef899f6769fe611b5557457a3802d5cfae060027745f6ba9ac21d4648f02ca8,0xfc97c2c87e3cc03beb5fa2facaa10bbd0d0da1af,0x08cac8952641d8fc526ec1ab4f2df826a5e7710f,0.01000,1438920403,0
16,0x09abf3b250969ab5a4bc80533f407dfaca950bbfd83395b78ec93562dd55e564,0xa89ac93b23370472daac337e9afdf642543f3e57,0x0000000000000000000000000000000000000000,100.00000,1438923647,0
17,0x09abf3b250969ab5a4bc80533f407dfaca950bbfd83395b78ec93562dd55e564,0xa89ac93b23370472daac337e9afdf642543f3e57,0x0000000000000000000000000000000000000000,100.00000,1438923647,0
23,0x85e4a08b2033222d8900552d2da73fc04d7f9aef2f73de1a13b0c5a63b5ec915,0xff68c2f39c4e7fe2f92c2dc2f8615b1bf070e0fa,0x0000000000000000000000000000000000000000,14.00000,1438925136,0
24,0x85e4a08b2033222d8900552d2da73fc04d7f9aef2f73de1a13b0c5a63b5ec915,0xff68c2f39c4e7fe2f92c2dc2f8615b1bf070e0fa,0x0000000000000000000000000000000000000000,14.00000,1438925136,0
43,0xfb3118914c770009654a36904897ac12a1d69b8479b7011ae2c389ff23ee4d90,0xc8ebccc5f5689fa8659d83713341e5ad19349448,0x0000000000000000000000000000000000000000,14.00000,1438930476,0
44,0xfb3118914c770009654a36904897ac12a1d69b8479b7011ae2c389ff23ee4d90,0xc8ebccc5f5689fa8659d83713341e5ad19349448,0x0000000000000000000000000000000000000000,14.00000,1438930476,0
47,0x0a085ffa148b92c77471191fcd1625ff5e0ffd3c080716e7ba9e40c11b2c0d35,0xc8ebccc5f5689fa8659d83713341e5ad19349448,0xff68c2f39c4e7fe2f92c2dc2f8615b1bf070e0fa,0.00001,1438931702,0
48,0x0a085ffa148b92c77471191fcd1625ff5e0ffd3c080716e7ba9e40c11b2c0d35,0xc8ebccc5f5689fa8659d83713341e5ad19349448,0xff68c2f39c4e7fe2f92c2dc2f8615b1bf070e0fa,0.00001,1438931702,0


In [4]:
def remove_duplicates_by_txHash(df):
    print("=== REMOVE DUPLICATES - KEEP FIRST ===")
    print(f"Sebelum: {len(df)} records")
    
    df_clean = df.drop_duplicates(subset=['TxHash'], keep='first')
    
    print(f"Sesudah: {len(df_clean)} records")
    print(f"Records dihapus: {len(df) - len(df_clean)}")
    
    return df_clean
df1 = remove_duplicates_by_txHash(df1)
df2 = remove_duplicates_by_txHash(df2)

=== REMOVE DUPLICATES - KEEP FIRST ===
Sebelum: 254973 records
Sesudah: 102149 records
Records dihapus: 152824
=== REMOVE DUPLICATES - KEEP FIRST ===
Sebelum: 6794521 records
Sesudah: 4604555 records
Records dihapus: 2189966


## b. Remove Nan

In [5]:
# Hapus Jika ditemukan ada Nan pada data
def remove_nan_rows_simple(df):
    # Simpan data original
    total_ori = len(df)
    
    # Hitung jumlah NaN sebelum dihapus
    nan_before = df[['From', 'To']].isna().any(axis=1).sum()
    
    # Hapus rows dengan NaN di From atau To
    df_clean = df.dropna(subset=['From', 'To'])
    
    # Hitung jumlah setelah dibersihkan
    total_clean = len(df_clean)
    nan_after = df_clean[['From', 'To']].isna().any(axis=1).sum()
    
    # Tampilkan hasil
    print("=== HASIL PEMBERSIHAN NaN ===")
    print(f"📊 Total data original: {total_ori:,}")
    print(f"🚫 Total rows dengan NaN: {nan_before:,}")
    print(f"✅ Total data setelah hapus NaN: {total_clean:,}")
    print(f"📈 Data yang dihapus: {total_ori - total_clean:,}")
    print(f"🎯 Persentase data hilang: {(total_ori - total_clean)/total_ori*100:.2f}%")
    print(f"🔍 Sisa NaN setelah cleaning: {nan_after}")
    
    return df_clean

df1 = remove_nan_rows_simple(df1)
df2 = remove_nan_rows_simple(df2)

=== HASIL PEMBERSIHAN NaN ===
📊 Total data original: 102,149
🚫 Total rows dengan NaN: 134
✅ Total data setelah hapus NaN: 102,015
📈 Data yang dihapus: 134
🎯 Persentase data hilang: 0.13%
🔍 Sisa NaN setelah cleaning: 0
=== HASIL PEMBERSIHAN NaN ===
📊 Total data original: 4,604,555
🚫 Total rows dengan NaN: 200
✅ Total data setelah hapus NaN: 4,604,355
📈 Data yang dihapus: 200
🎯 Persentase data hilang: 0.00%
🔍 Sisa NaN setelah cleaning: 0


# 2. Build Basic Graph

In [6]:
# Build Basic Directed Graph Representation
import networkx as nx

G1 = nx.from_pandas_edgelist(
    df1,
    source='From',
    target='To',
    edge_attr=['Value', 'TimeStamp', 'isError'], # Atribut untuk setiap edge
    create_using=nx.DiGraph()
)
print(f"Graph 1 berhasil dibuat. Jumlah Node: {G1.number_of_nodes()}, Jumlah Edge: {G1.number_of_edges()}")

G2 = nx.from_pandas_edgelist(
    df2,
    source='From',
    target='To',
    edge_attr=['Value', 'TimeStamp', 'isError'], # Atribut untuk setiap edge
    create_using=nx.DiGraph()
)

print(f"Graph 2 berhasil dibuat. Jumlah Node: {G2.number_of_nodes()}, Jumlah Edge: {G2.number_of_edges()}")

Graph 1 berhasil dibuat. Jumlah Node: 44877, Jumlah Edge: 61350
Graph 2 berhasil dibuat. Jumlah Node: 1329726, Jumlah Edge: 2161494


In [7]:
# Cek apakah ada Self Loops
print(f"Number of self-loops: {nx.number_of_selfloops(G1)}")
print(f"Number of self-loops: {nx.number_of_selfloops(G2)}")

Number of self-loops: 40
Number of self-loops: 942


# 3. Build Node Atribut

In [8]:
# Bangun Atribut Node
from datetime import datetime

def build_atribut(G):
    # Hitung metrik global
    pagerank_scores = nx.pagerank(G)
    clustering_coeffs = nx.clustering(G)
    
    for node in G.nodes():
        try:
            # 1. Fitur dasar degree
            G.nodes[node]['in_degree'] = G.in_degree(node)
            G.nodes[node]['out_degree'] = G.out_degree(node)
            
            in_edges = list(G.in_edges(node, data=True))
            out_edges = list(G.out_edges(node, data=True))
            
            # 2. Hitung jumlah transaksi fraud
            in_fraud_count = sum(1 for _, _, d in in_edges if d.get('isError', 0) == 1)
            out_fraud_count = sum(1 for _, _, d in out_edges if d.get('isError', 0) == 1)
            
            G.nodes[node]['in_fraud_count'] = in_fraud_count
            G.nodes[node]['out_fraud_count'] = out_fraud_count
            
            # 3. Hitung persentase fraud
            G.nodes[node]['perc_in_fraud'] = in_fraud_count / G.nodes[node]['in_degree'] if G.nodes[node]['in_degree'] > 0 else 0
            G.nodes[node]['perc_out_fraud'] = out_fraud_count / G.nodes[node]['out_degree'] if G.nodes[node]['out_degree'] > 0 else 0
            
            # 4. Statistik nilai transaksi (in)
            in_values = [d.get('Value', 0) for _, _, d in in_edges if 'Value' in d]
            out_values = [d.get('Value', 0) for _, _, d in out_edges if 'Value' in d]
            
            G.nodes[node]['std_value_in'] = np.std(in_values) if in_values else 0
            G.nodes[node]['min_value_in'] = np.min(in_values) if in_values else 0
            G.nodes[node]['max_value_in'] = np.max(in_values) if in_values else 0
            G.nodes[node]['med_value_in'] = np.median(in_values) if in_values else 0
            G.nodes[node]['avg_value_in'] = np.mean(in_values) if in_values else 0
            
            # 5. Statistik nilai transaksi (out)
            G.nodes[node]['std_value_out'] = np.std(out_values) if out_values else 0
            G.nodes[node]['min_value_out'] = np.min(out_values) if out_values else 0
            G.nodes[node]['max_value_out'] = np.max(out_values) if out_values else 0
            G.nodes[node]['med_value_out'] = np.median(out_values) if out_values else 0
            G.nodes[node]['avg_value_out'] = np.mean(out_values) if out_values else 0
            
            # Helper function untuk menghitung time differences
            def calculate_time_differences(edges, fraud_only=False):
                times = []
                for _, _, d in edges:
                    if 'TimeStamp' in d:
                        if fraud_only and d.get('isError', 0) != 1:
                            continue
                        try:
                            times.append(datetime.strptime(d['TimeStamp'], '%Y-%m-%d %H:%M:%S'))
                        except (ValueError, TypeError):
                            continue
                
                if len(times) > 1:
                    times.sort()
                    time_diffs = [(times[i] - times[i-1]).total_seconds() 
                                for i in range(1, len(times))]
                    return np.mean(time_diffs)
                return 0
            
            # 6-9. Hitung time differences
            G.nodes[node]['tdif_fraud_in'] = calculate_time_differences(in_edges, fraud_only=True)
            G.nodes[node]['tdif_fraud_out'] = calculate_time_differences(out_edges, fraud_only=True)
            G.nodes[node]['tdif_trx_in'] = calculate_time_differences(in_edges, fraud_only=False)
            G.nodes[node]['tdif_trx_out'] = calculate_time_differences(out_edges, fraud_only=False)
            
            # 10. Hitung TOTAL NILAI transaksi fraud
            in_fraud_values = [d.get('Value', 0) for _, _, d in in_edges if d.get('isError', 0) == 1 and 'Value' in d]
            out_fraud_values = [d.get('Value', 0) for _, _, d in out_edges if d.get('isError', 0) == 1 and 'Value' in d]
            
            G.nodes[node]['total_fraud_value_in'] = sum(in_fraud_values) if in_fraud_values else 0
            G.nodes[node]['total_fraud_value_out'] = sum(out_fraud_values) if out_fraud_values else 0
            
            # Fitur tambahan
            G.nodes[node]['pagerank'] = pagerank_scores.get(node, 0)
            G.nodes[node]['clustering_coefficient'] = clustering_coeffs.get(node, 0)
            G.nodes[node]['total_value_received'] = sum(in_values) if in_values else 0
            G.nodes[node]['total_value_sent'] = sum(out_values) if out_values else 0
            G.nodes[node]['balance'] = G.nodes[node]['total_value_received'] - G.nodes[node]['total_value_sent']
            
        except Exception as e:
            print(f"Error processing node {node}: {str(e)}")
            # Lanjut ke node berikutnya
    
    return G  # Kembalikan graph setelah semua node diproses


G1 = build_atribut(G1)
G2 = build_atribut(G2)

In [9]:
# Buat list semua node dengan total fraud count
def show_graph_atribut(G,num=30):
    fraud_data = []
    for node in G.nodes():
        total_fraud = G.nodes[node].get('in_fraud_count', 0) + G.nodes[node].get('out_fraud_count', 0)
        fraud_data.append({
            'Node': node,
            'InDegree' : G.nodes[node].get('in_degree', 0),
            'OutDegree' : G.nodes[node].get('out_degree', 0),
            'InFraud': G.nodes[node].get('in_fraud_count', 0),
            'OutFraud': G.nodes[node].get('out_fraud_count', 0),
            'InFraudValue': G.nodes[node].get('total_fraud_value_in', 0),
            'OutFraudValue': G.nodes[node].get('total_fraud_value_out', 0),
            'TotalFraud': total_fraud,
            'PercFraud': (G.nodes[node].get('in_fraud_count', 0) / G.nodes[node].get('in_degree', 1) if G.nodes[node].get('in_degree', 0) > 0 else 0) +
                          (G.nodes[node].get('out_fraud_count', 0) / G.nodes[node].get('out_degree', 1) if G.nodes[node].get('out_degree', 0) > 0 else 0)
        })
    
    # Konversi ke DataFrame dan urutkan
    df_fraud = pd.DataFrame(fraud_data)
    df_fraud = df_fraud.sort_values('InFraud', ascending=False).head(num)
    
    # Format persentase
    df_fraud['PercFraud'] = df_fraud['PercFraud'].map('{:.2%}'.format)

    print()
    print(f"{num} Node dengan Fraud Count Terbanyak:")
    display(df_fraud.set_index('Node'))

show_graph_atribut(G1,30)
show_graph_atribut(G2,30)


30 Node dengan Fraud Count Terbanyak:


,InDegree,OutDegree,InFraud,OutFraud,InFraudValue,OutFraudValue,TotalFraud,PercFraud
Node,,,,,,,,
0x903bb9cd3a276d8f18fa6efed49b9bc52ccf06e5,1610,1,371,0,0.019400,0.000000,371,23.04%
0xe8868e87aaa4a0d0751691f9f33b0e5da7127039,2453,0,213,0,0.001120,0.000000,213,8.68%
0xe0787aabbd6ee01e55f98647673552885d54eb06,108,1,56,0,0.001400,0.000000,56,51.85%
0xf2bad87c0d0ea8bda69c722368df4f79d92ee6c9,339,1,35,0,0.118920,0.000000,35,10.32%
0x70305b080efc49eb5dfb9bda78aea516c398f804,202,1,32,0,0.309709,0.000000,32,15.84%
0xc1e6e4d9fc0b7e555bb8634bcacc9a1067bec039,66,37,30,0,2.755131,0.000000,30,45.45%
0xbfa82fbe0e66d8e2b7dcc16328db9ecd70533d13,422,1,17,0,0.011905,0.000000,17,4.03%
0x8d12a197cb00d4747a1fe03395095ce2a5cc6819,103,71,14,0,0.000000,0.000000,14,13.59%
0x9e19462787be36e5e3676ad2428b26599bf9866c,61,0,11,0,0.000000,0.000000,11,18.03%



30 Node dengan Fraud Count Terbanyak:


,InDegree,OutDegree,InFraud,OutFraud,InFraudValue,OutFraudValue,TotalFraud,PercFraud
Node,,,,,,,,
0x741fc999f5b62c80831cf659aed04c64ac8ef24e,7254,0,5893,0,42079.044115,0.0,5893,81.24%
0xa66d83716c7cfe425b44d0f7ef92de263468fb3d,5609,0,5442,0,98471.090659,0.0,5442,97.02%
0x6267b5376c809445c9432bd9f14a3808b00eae2c,5941,0,4959,0,12223.010228,0.0,4959,83.47%
0x12444b6ec62e616ebc8a23e56e61f8f4c6da610c,5151,0,1874,0,4593.838215,0.0,1874,36.38%
0x9d9bcdd249e439aaab545f59a33812e39a8e3072,2848,0,1798,0,16701.331834,0.0,1798,63.13%
0x91c94bee75786fbbfdcfefba1102b68f48a002f4,2106,0,1716,0,11491.948583,0.0,1716,81.48%
0xa94c2707e233e5d63f46011a4a158cd2452d1d86,2892,0,1692,0,4465.506607,0.0,1692,58.51%
0x99287f6a84d56fc3bb2ad95a4bbe783403f825f0,2112,0,1529,0,3559.316970,0.0,1529,72.40%
0x01bbec6573ed7eca0f307a10d2b4ceb669816b4a,5893,0,1398,0,1067.644817,0.0,1398,23.72%


# 3. Node Labelling

In [10]:
def label_fraud_nodes(G):
    print()
    print("=== NODE FRAUD LABELLING ===")
    fraud_count = 0
    total_nodes = G.number_of_nodes()
    
    for node in G.nodes():
        # Default value 0 (non-fraud)
        is_fraud = 0
        
        # Cek jika total_fraud_value_in > 0
        if 'total_fraud_value_in' in G.nodes[node]:
            fraud_value = G.nodes[node]['total_fraud_value_in']
            if fraud_value > 0:
                is_fraud = 1
                fraud_count += 1
        
        # Tambahkan atribut is_fraud_node
        G.nodes[node]['is_fraud_node'] = is_fraud
    
    print(f"📊 Total nodes: {total_nodes:,}")
    print(f"🔴 Fraud nodes: {fraud_count:,}")
    print(f"🟢 Non-fraud nodes: {total_nodes - fraud_count:,}")
    print(f"🎯 Persentase fraud: {fraud_count/total_nodes*100:.2f}%")
    
    return G

G1 = label_fraud_nodes(G1)
G2 = label_fraud_nodes(G2)


=== NODE FRAUD LABELLING ===
📊 Total nodes: 44,877
🔴 Fraud nodes: 54
🟢 Non-fraud nodes: 44,823
🎯 Persentase fraud: 0.12%

=== NODE FRAUD LABELLING ===
📊 Total nodes: 1,329,726
🔴 Fraud nodes: 4,508
🟢 Non-fraud nodes: 1,325,218
🎯 Persentase fraud: 0.34%


# 4. REMOVE Isolated Non-Fraud Nodes

In [11]:
# Drop Node dengan kriteria :
# 1. Non-Fraud Node
# 2. number of degree = 1 

def remove_isolated_non_fraud_nodes(G):
    print("\n=== REMOVE ISOLATED NON-FRAUD NODES ===")
    
    nodes_before = G.number_of_nodes()
    edges_before = G.number_of_edges()
    
    # Hitung fraud nodes sebelum pembersihan
    fraud_nodes_before = sum(1 for node in G.nodes() if G.nodes[node].get('is_fraud_node', 0) == 1)
    
    # Identifikasi nodes yang memenuhi kriteria penghapusan
    nodes_to_remove = []
    
    for node in G.nodes():
        # Cek kondisi: non-fraud (is_fraud_node=0) dan degree=1
        is_non_fraud = G.nodes[node].get('is_fraud_node', 0) == 0
        degree = G.degree(node)
        
        if is_non_fraud and degree == 1:
            nodes_to_remove.append(node)
    
    # Hapus nodes
    G.remove_nodes_from(nodes_to_remove)
    
    nodes_after = G.number_of_nodes()
    edges_after = G.number_of_edges()
    
    # Hitung fraud nodes setelah pembersihan
    fraud_nodes_after = sum(1 for node in G.nodes() if G.nodes[node].get('is_fraud_node', 0) == 1)
    
    # Reporting
    print(f"📊 HASIL PEMBERSIHAN:")
    print(f"   • Nodes sebelum: {nodes_before:,}")
    print(f"   • Nodes sesudah: {nodes_after:,}")
    print(f"   • Nodes dihapus: {len(nodes_to_remove):,}")
    print(f"   • Edges sebelum: {edges_before:,}")
    print(f"   • Edges sesudah: {edges_after:,}")
    print(f"   • Edges terhapus: {edges_before - edges_after:,}")
    print(f"   • Persentase nodes dihapus: {len(nodes_to_remove)/nodes_before*100:.2f}%")
    
    print(f"\n🔴 FRAUD NODE STATISTICS:")
    print(f"   • Fraud nodes sebelum: {fraud_nodes_before:,}")
    print(f"   • Fraud nodes sesudah: {fraud_nodes_after:,}")
    print(f"   • Persentase fraud sebelum: {fraud_nodes_before/nodes_before*100:.2f}%")
    print(f"   • Persentase fraud sesudah: {fraud_nodes_after/nodes_after*100:.2f}%")
    
    return G

G1 = remove_isolated_non_fraud_nodes(G1)
G2 = remove_isolated_non_fraud_nodes(G2)


=== REMOVE ISOLATED NON-FRAUD NODES ===
📊 HASIL PEMBERSIHAN:
   • Nodes sebelum: 44,877
   • Nodes sesudah: 7,613
   • Nodes dihapus: 37,264
   • Edges sebelum: 61,350
   • Edges sesudah: 24,108
   • Edges terhapus: 37,242
   • Persentase nodes dihapus: 83.04%

🔴 FRAUD NODE STATISTICS:
   • Fraud nodes sebelum: 54
   • Fraud nodes sesudah: 54
   • Persentase fraud sebelum: 0.12%
   • Persentase fraud sesudah: 0.71%

=== REMOVE ISOLATED NON-FRAUD NODES ===
📊 HASIL PEMBERSIHAN:
   • Nodes sebelum: 1,329,726
   • Nodes sesudah: 324,151
   • Nodes dihapus: 1,005,575
   • Edges sebelum: 2,161,494
   • Edges sesudah: 1,155,928
   • Edges terhapus: 1,005,566
   • Persentase nodes dihapus: 75.62%

🔴 FRAUD NODE STATISTICS:
   • Fraud nodes sebelum: 4,508
   • Fraud nodes sesudah: 4,508
   • Persentase fraud sebelum: 0.34%
   • Persentase fraud sesudah: 1.39%


In [12]:
# Convert to Integer Labe
G1 = nx.convert_node_labels_to_integers(G1)
G2 = nx.convert_node_labels_to_integers(G2)

# Split 20% Test
from sklearn.model_selection import train_test_split

def stratified_split_graph_nodes(G, label_key="label", test_size=0.2, random_state=42):
    
    # Ambil semua node ID
    nodes = list(G.nodes())

    # Ambil label tiap node
    labels = [G.nodes[n][label_key] for n in nodes]

    # Stratified split
    train_nodes, test_nodes = train_test_split(
        nodes,
        test_size=test_size,
        stratify=labels,
        random_state=random_state
    )

    return test_nodes

G1_test_nodes = stratified_split_graph_nodes(
    G1,
    label_key="is_fraud_node",
    test_size=0.2
)

G2_test_nodes = stratified_split_graph_nodes(
    G2,
    label_key="is_fraud_node",
    test_size=0.2
)

In [13]:
# Simpan Graph Dataset
import pickle
import os
import json

def save_dataset(G, filename):
    os.makedirs('/kaggle/working/graph_data', exist_ok=True)
    with open(f'/kaggle/working/graph_data/{filename}', 'wb') as f:
        pickle.dump(G, f)

    print(f"{filename} berhasil diekspor!")
    print(f"Jumlah nodes: {G.number_of_nodes()}")
    print(f"Jumlah edges: {G.number_of_edges()}")

def save_nodes_to_json(nodes, filename="nodes.json"):
    with open(filename, "w") as f:
        json.dump(nodes, f, indent=2)

save_dataset(G1, 'first_graph.pkl')
save_nodes_to_json(G1_test_nodes, "first_test_graph.json")

save_dataset(G2, 'second_graph.pkl')
save_nodes_to_json(G2_test_nodes, 'second_test_graph.json')

first_graph.pkl berhasil diekspor!
Jumlah nodes: 7613
Jumlah edges: 24108
second_graph.pkl berhasil diekspor!
Jumlah nodes: 324151
Jumlah edges: 1155928
